# Modelling

First version of a model predicting the number of `Rings` of an abalone. We compare a very simple baseline with 2 ML models, and track every experiment with MLflow.

To compare the runs in the MLflow UI, run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the `notebooks/` folder and open http://127.0.0.1:5000.

# Imports

In [3]:
import mlflow
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

# Data

We remove the 4 `Height` outliers identified in the EDA (`Height = 0` and `Height > 0.4`) and one-hot encode `Sex`.

In [4]:
df = pd.read_csv("../data/abalone.csv")
df = df[(df["Height"] > 0) & (df["Height"] <= 0.4)]
df = pd.get_dummies(df, columns=["Sex"], drop_first=True, dtype=int)

X = df.drop(columns="Rings")
y = df["Rings"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (3338, 9), Test: (835, 9)


# Models

- **Baseline**: always predicts the mean number of rings of the train set. Any useful model must beat it.
- **Linear regression**: simple and interpretable model.
- **Random forest**: non-linear model, which can capture the curved relationships seen in the EDA (e.g. `Height` vs `Whole weight`).

In [ ]:
models = {
    "baseline_mean": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

# Relative path: MLflow's default absolute path breaks when the folder path contains spaces
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("abalone-age-prediction")

results = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = {
            "rmse": root_mean_squared_error(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred),
            "r2": r2_score(y_test, y_pred),
        }
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)

    results.append({"model": name, **metrics})

pd.DataFrame(results).set_index("model").round(3)

# Conclusions

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Baseline (mean) | 3.46 | 2.54 | 0.00 |
| Linear regression | 2.23 | 1.58 | 0.585 |
| Random forest | 2.25 | 1.56 | 0.579 |

- **Both ML models clearly beat the baseline**: the RMSE drops from 3.46 to about 2.2 rings (-36%), and they explain about 58% of the variance of `Rings`.
- **The random forest does not beat the linear regression**: its scores are almost identical (slightly better MAE, slightly worse RMSE and R²), while it is slower and much harder to interpret.
- **We keep the linear regression** as our first model: same performance, simpler, and enough for this project.
- **The error is still about 2 rings** (≈ 2 years) on average: physical measurements alone cannot predict the age precisely, which matches the EDA (best correlation with `Rings` is only 0.63).